# 01 - EDA de fuentes de datos | RADAR Cibest

**Fase ASUM-DM:** 2 - Entendimiento de datos
**Co-lideres:** Jhon y Laura  
**Fecha:** Marzo - Abril 2026

Este notebook explora la disponibilidad y calidad de las fuentes de datos antes de ejecutar el pipeline en produccion.

Objetivos:
1. Validar conectividad con cada API
2. Verificar cobertura por pais y variable
3. Identificar variables con datos incompletos
4. Generar reporte preliminar de calidad

In [151]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.utils import load_all_configs, setup_logger

configs = load_all_configs()
setup_logger(configs['settings'].get('logging'))
print(f"Proyecto: {configs['settings']['project']['name']}")
print(f"Paises en alcance: {len(configs['settings']['countries'])}")
print(f"Dimensiones: {configs['settings']['project']['dimensions']}")

Proyecto: RADAR Cibest
Paises en alcance: 30
Dimensiones: ['macro', 'financial', 'institutional', 'digital_tech', 'proximity']


## 1. Inventario de variables por dimension

In [152]:
from src.utils import get_variable_catalog

catalog = get_variable_catalog(configs['variables'])
df_catalog = pd.DataFrame.from_dict(catalog, orient='index')
print(f"Total variables: {len(df_catalog)}")
print(f"\nVariables por dimension:")
print(df_catalog.groupby('dimension').size())
print(f"\nVariables por fuente:")
print(df_catalog.groupby('source').size())

Total variables: 45

Variables por dimension:
dimension
digital_tech      7
financial         9
institutional     8
macro            12
proximity         9
dtype: int64

Variables por fuente:
source
complementary                     10
damodaran_country_risk_premium     1
world_bank                        34
dtype: int64


## 2. Test de extraccion - Banco Mundial

In [153]:
from src.data_extraction.world_bank import fetch_indicator

df_test = fetch_indicator(
    indicator_code='NY.GDP.PCAP.CD',
    countries=['COL', 'MEX', 'CHL', 'ESP']
)
df_test.head()

,country_iso3,year,variable,value
0,CHL,2024,NY.GDP.PCAP.CD,16709.889397
1,COL,2024,NY.GDP.PCAP.CD,7919.208868
2,ESP,2024,NY.GDP.PCAP.CD,35326.768307
3,MEX,2024,NY.GDP.PCAP.CD,14185.781225


## 3. Pipeline completo de extraccion

In [154]:
import importlib
import inspect

import src.utils as utils
import src.data_extraction.world_bank as world_bank
import src.data_extraction.complementary as complementary
import src.data_extraction.pipeline as pipeline
import src.scoring.hybrid_scorer as hybrid_scorer

importlib.invalidate_caches()

importlib.reload(world_bank)
importlib.reload(complementary)
importlib.reload(pipeline)
importlib.reload(hybrid_scorer)
importlib.reload(utils)


from src.utils import (
    load_all_configs,
    get_world_bank_variable_catalog,
    infer_world_bank_db,
)


from src.data_extraction.pipeline import run_extraction

configs = load_all_configs()

print("Tiene fetch_indicator_history:", hasattr(world_bank, "fetch_indicator_history"))


Tiene fetch_indicator_history: True


In [155]:
print("complementary file:", complementary.__file__)
print("pipeline file:", pipeline.__file__)
print("fetch_colombian_outbound_travel starts at line:",
      inspect.getsourcelines(complementary.fetch_colombian_outbound_travel)[1])
print("save_raw_data starts at line:",
      inspect.getsourcelines(complementary.save_raw_data)[1])


complementary file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\data_extraction\complementary.py
pipeline file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\data_extraction\pipeline.py
fetch_colombian_outbound_travel starts at line: 261
save_raw_data starts at line: 392


In [156]:
master, coverage = run_extraction(configs=configs, save_intermediate=True)

print(f"Master shape: {master.shape}")
print(f"Cobertura promedio: {coverage['pct_cobertura'].mean():.1f}%")


2026-06-16 16:58:04 | INFO     | src.data_extraction.pipeline:run_extraction:116 | Fuente world_bank en modo latest_available
2026-06-16 16:58:04 | INFO     | src.data_extraction.world_bank:fetch_all_indicators:254 | Indicadores World Bank agrupados por base: {2: 24, 32: 3, 3: 6, 28: 1}
2026-06-16 16:58:04 | INFO     | src.data_extraction.world_bank:fetch_all_indicators:268 | Extrayendo indicadores World Bank: db=2, n_indicadores=24
2026-06-16 16:58:04 | INFO     | src.data_extraction.world_bank:fetch_all_indicators:268 | Extrayendo indicadores World Bank: db=32, n_indicadores=3
2026-06-16 16:58:05 | INFO     | src.data_extraction.world_bank:fetch_all_indicators:268 | Extrayendo indicadores World Bank: db=3, n_indicadores=6
2026-06-16 16:58:05 | INFO     | src.data_extraction.world_bank:fetch_all_indicators:268 | Extrayendo indicadores World Bank: db=28, n_indicadores=1
2026-06-16 16:58:05 | INFO     | src.data_extraction.pipeline:run_extraction:144 | Anexando histórico específico de g

Master shape: (1288, 5)
Cobertura promedio: 91.0%


In [157]:
print("utils file:", utils.__file__)
print("Allowed DBs:", utils.WORLD_BANK_ALLOWED_DBS)

utils file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\utils.py
Allowed DBs: {32, 2, 3, 28}


1. Qué significa cada línea
    World Bank guardado
    Plain TextWorld Bank raw guardado: ...\data\raw\world_bank_20260513.parquetMostrar más líneas
    Significa que world_bank.py extrajo los indicadores de:

    WDI / Findex db=2
    WGI db=3
    GFDD db=32

    y los guardó en:
    data/raw/world_bank_20260513.parquetMostrar más líneas

    Damodaran descargado y guardado
    Plain TextDescargando Country Risk Premium desde Damodaran...Damodaran raw guardado: ...\damodaran_country_risk_premium_20260513.parquetMostrar más líneas
    Significa que el script de Damodaran descargó el Excel de NYU Stern, procesó la prima de riesgo país y guardó el resultado.
    El warning:
    InsecureRequestWarning: Unverified HTTPS requestMostrar más líneas
    no es un error. Solo indica que estás descargando con verify_ssl=False. El pipeline continuó normalmente.

    Complementary guardado
    Plain TextComplementary raw guardado: ...\complementary_20260513.parquetMostrar más líneas
    Significa que complementary.py sí corrió y guardó las variables complementarias, como CEPII, Hofstede, Heritage o salidas de colombianos, según lo que esté activo en data_sources.yaml.

    Master shape
    Master shape: (916, 5)Mostrar más líneas
    Significa que el DataFrame maestro tiene:
    916 filas5 columnasMostrar más líneas
    Normalmente las columnas son:
    Plain Textcountry_iso3yearvariablevaluesourceMostrar más líneas
    Cada fila representa una observación tipo:
    país - año - variable - valor - fuenteMostrar más líneas

2. Qué significa cobertura promedio 87.2%
    Plain TextCobertura promedio: 87.2%Mostrar más líneas
    Esto quiere decir que, en promedio, cada país tiene datos para el 87.2% de las variables esperadas en variables.yaml.
    No quiere decir que el pipeline falló. Quiere decir que el universo de variables esperadas es más grande que los datos efectivamente disponibles para todos los países.
    La cobertura probablemente se está calculando así:
    Pythonpct_cobertura = variables_disponibles_para_el_pais / total_variables_esperadasMostrar más líneas
    Por ejemplo, si el catálogo tiene 31 variables y un país tiene datos para 27:
    27 / 31 = 87.1%Mostrar más líneas

3. Por qué no es 100%
    Hay varias razones normales:
    1. No todos los países tienen datos para todas las variables World Bank
        Algunos indicadores no tienen dato reciente o disponible para todos los países, especialmente en:

        países del Caribe,
        economías pequeñas,
        Cuba,
        Venezuela,
        algunos indicadores GFDD,
        algunos indicadores Findex.


    2. Algunas fuentes complementarias tienen cobertura parcial
        Variables como:

        Hofstede,
        Heritage EFI,
        CEPII,
        salidas de colombianos,
        Country Risk Premium,

        pueden no tener dato para todos los países del universo.
        Por ejemplo, Hofstede suele tener menos cobertura que World Bank. Heritage también puede no mapear todos los países si el nombre del país no coincide con el diccionario HERITAGE_COUNTRY_TO_ISO3.

    3. Damodaran puede no cubrir todos los países
        El Excel de Damodaran no necesariamente tiene prima de riesgo para todos los países del alcance, o puede tener nombres que pycountry no logra convertir automáticamente a ISO3.

    4. El denominador usa todo el catálogo de variables
        El reporte de cobertura compara cada país contra todas las variables de variables.yaml, no solo contra las que tienen datos disponibles en alguna fuente.
        Entonces, si hay una variable declarada en variables.yaml pero no llega desde ninguna fuente para un país, baja la cobertura.

    5. Posibles diferencias entre nombres de variables
        Si una fuente genera una variable con un nombre distinto al declarado en variables.yaml, el dato existe en master, pero el reporte de cobertura lo cuenta como faltante.
            Ejemplo:
            Plain TextDeclarado en variables.yaml: cultural_distance_hofstedeGenerado por complementary.py: hofstede_pdi, hofstede_idv, ...Mostrar más líneas
            Si el catálogo espera una variable y el extractor entrega otra, baja la cobertura.

In [158]:
coverage.sort_values("pct_cobertura") #.head(10)

,country_iso3,n_variables_total,n_variables_disponibles,pct_cobertura,variables_faltantes
29,CUB,45,23,51.11,account_ownership; atms_per_100k_adults; bank_...
28,GUY,45,32,71.11,account_ownership; bank_capital_rwa; bank_npl_...
27,HTI,45,33,73.33,bank_capital_rwa; bank_npl_ratio; country_risk...
26,BHS,45,34,75.56,account_ownership; bank_capital_rwa; bank_npl_...
25,BRB,45,35,77.78,account_ownership; bank_capital_rwa; digital_p...
24,BLZ,45,37,82.22,bank_capital_rwa; hofstede_idv; hofstede_ivr; ...
23,SUR,45,37,82.22,account_ownership; bank_capital_rwa; bank_npl_...
22,NIC,45,37,82.22,hofstede_idv; hofstede_ivr; hofstede_lto; hofs...
21,JAM,45,41,91.11,bank_npl_ratio; hofstede_ivr; hofstede_lto; tr...
20,HND,45,41,91.11,hofstede_ivr; hofstede_lto; public_debt_gdp; s...


In [159]:
master.groupby("year").size().sort_index()

year
1996      1
1999      2
2000      4
2001      3
2002      1
2005      1
2006      3
2007      1
2008      2
2009      1
2010      3
2011      1
2013      2
2014      3
2015      4
2016      4
2017      8
2018      2
2019      5
2020    212
2021     63
2022     94
2023    106
2024    645
2025     60
2026     57
dtype: int64

In [160]:
vars_check = [
    "gdp_nominal",
    "gdp_per_capita_ppp",
    "gdp_growth",
    "inflation_rate",
    "unemployment_rate",
    "domestic_credit_private_gdp",
    "account_ownership",
    "regulatory_quality",
    "government_effectiveness",
    "rule_of_law",
    "internet_users_pct",
    "secure_internet_servers_per_million",
]

(
    master[master["variable"].isin(vars_check)]
    .groupby(["variable", "year"])
    .size()
    .reset_index(name="n")
    .sort_values(["variable", "year"])
)

,variable,year,n
0,account_ownership,2017,1
1,account_ownership,2021,1
2,account_ownership,2024,23
3,domestic_credit_private_gdp,2008,1
4,domestic_credit_private_gdp,2015,1
5,domestic_credit_private_gdp,2023,1
6,domestic_credit_private_gdp,2024,26
7,gdp_growth,2022,30
8,gdp_growth,2023,30
9,gdp_growth,2024,30


In [161]:
master[master["variable"] == "gdp_growth"].sort_values(
    ["country_iso3", "year"]
).head(20)

,country_iso3,year,variable,value,source
15,ARG,2022,gdp_growth,6.020745,world_bank
16,ARG,2023,gdp_growth,-1.855788,world_bank
17,ARG,2024,gdp_growth,-1.342931,world_bank
57,BHS,2022,gdp_growth,10.877901,world_bank
58,BHS,2023,gdp_growth,3.048242,world_bank
59,BHS,2024,gdp_growth,3.378666,world_bank
96,BLZ,2022,gdp_growth,9.250513,world_bank
97,BLZ,2023,gdp_growth,0.500128,world_bank
98,BLZ,2024,gdp_growth,3.504664,world_bank
136,BOL,2022,gdp_growth,3.747418,world_bank


In [162]:
(
    master[master["variable"] == "gdp_growth"]
    .groupby("year")
    .size()
    .sort_index()
)

year
2022    30
2023    30
2024    30
dtype: int64

In [163]:
master[master["variable"] == "gdp_growth"]["value"].describe()

count    90.000000
mean      4.570314
std       8.770089
min      -4.169634
25%       1.734381
50%       3.078248
75%       4.303408
max      63.334634
Name: value, dtype: float64

In [164]:
df_g = master[master["variable"] == "gdp_growth"].copy()
df_g["year"] = pd.to_numeric(df_g["year"], errors="coerce")
df_g["value"] = pd.to_numeric(df_g["value"], errors="coerce")

print(df_g.shape)
print(df_g["year"].value_counts().sort_index())
print(df_g["value"].describe())
print(df_g.groupby("country_iso3")["value"].mean().sort_values())

(90, 5)
year
2022    30
2023    30
2024    30
Name: count, dtype: int64
count    90.000000
mean      4.570314
std       8.770089
min      -4.169634
25%       1.734381
50%       3.078248
75%       4.303408
max      63.334634
Name: value, dtype: float64
country_iso3
HTI    -2.571834
CUB    -0.405484
ARG     0.940676
TTO     1.612309
BOL     1.713476
CHL     1.773224
PER     1.903244
ECU     1.951802
SUR     2.181257
CAN     2.424192
USA     2.730978
URY     2.778870
MEX     2.830242
JAM     2.883536
SLV     3.031931
PRY     3.141742
COL     3.212996
BRA     3.225888
HND     3.757793
GTM     3.789847
NIC     3.855291
ESP     4.095476
DOM     4.127926
BLZ     4.418435
CRI     4.661546
VEN     5.767533
BHS     5.768269
PAN     6.984254
BRB     7.541027
GUY    46.982985
Name: value, dtype: float64


In [165]:
from pathlib import Path

# Crear carpeta de reportes si no existe
output_dir = Path("data/reports")
output_dir.mkdir(parents=True, exist_ok=True)

# Ruta de salida
output_file = output_dir / "master_radar_cibest.xlsx"

# Exportar master a Excel
master.to_excel(output_file, index=False, sheet_name="master")

print(f"Master exportada correctamente en: {output_file}")
print(f"Shape exportado: {master.shape}")

Master exportada correctamente en: data\reports\master_radar_cibest.xlsx
Shape exportado: (1288, 5)


In [166]:
# from ydata_profiling import ProfileReport
# master_wide = (
#     master
#     .sort_values("year")
#     .drop_duplicates(subset=["country_iso3", "variable"], keep="last")
#     .pivot(index="country_iso3", columns="variable", values="value")
#     .reset_index()
# )

# profile = ProfileReport(
#     master_wide,
#     title="Reporte de Perfilamiento - Master Dataset",
#     explorative=True,
#     minimal=False,
# )

# profile.to_file("master_profile_report.html")